# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'product page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'related platform',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'Endpoints product', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
2 days ago
•
174k
•
3.17k
openai/privacy-filter
Updated
7 days ago
•
57.7k
•
1.04k
Qwen/Qwen3.6-27B
Updated
5 days ago
•
509k
•
973
deepseek-ai/DeepSeek-V4-Flash
Updated
2 days ago
•
96.9k
•
827
unsloth/Qwen3.6-27B-GGUF
Updated
7 days ago
•
702k
•
481
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
246
ML Intern
🤖
246
Explore ML concepts and get help with a virtual intern
Running
on
Zero
MCP
2.43k
Wan2.2 14B Preview
🐌
2.43k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Fea

In [25]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n2 days ago\n•\n174k\n•\n3.17k\nopenai/privacy-filter\nUpdated\n7 days ago\n•\n57.7k\n•\n1.04k\nQwen/Qwen3.6-27B\nUpdated\n5 days ago\n•\n509k\n•\n973\ndeepseek-ai/DeepSeek-V4-Flash\nUpdated\n2 days ago\n•\n96.9k\n•\n827\nunsloth/Qwen3.6-27B-GGUF\nUpdated\n7 days ago\n•\n702k\n•\n481\nBrowse 2M+ models\nSpaces\nRunning\non\nCPU Upgrade\n246\nML Intern

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face: Building the Future of AI, Together

---

## About Hugging Face

Hugging Face is the premier collaboration platform for the machine learning (ML) community. It serves as a central hub where ML engineers, scientists, and enthusiasts can share, explore, and experiment with an extensive repository of open-source ML resources including:

- **Models:** Access and browse over 2 million ML models across text, image, video, audio, and 3D modalities.
- **Datasets:** Utilize a vast catalog of over 500,000 datasets for a variety of tasks.
- **Spaces:** Host and deploy ML applications seamlessly.
- **Buckets:** Manage data storage and usage efficiently.

By empowering the next generation of machine learning practitioners, Hugging Face drives the creation of an open and ethical AI future. At the heart of the AI revolution, the platform champions community-led innovation with open-source tools and a cutting-edge science team.

---

## Why Choose Hugging Face?

- **Collaborative Ecosystem:** Host unlimited public models, datasets, and applications — making it easier to experiment, learn, and share your work.
- **Community Powered:** Join a fast-growing global community contributing to some of the most popular open-source ML libraries and models.
- **Multi-Modal Flexibility:** Work across a variety of data formats — text, image, video, audio, and even 3D.
- **Open and Ethical AI:** Foster transparency and ethical considerations through open source.
- **Developer Growth:** Build your ML profile and portfolio openly to accelerate your career and influence.

---

## Enterprise Solutions

Hugging Face offers scalable plans tailored to businesses of all sizes:

### Team Plan
- Starting at $20/user/month
- Collaboration designed for small to mid-size teams needing advanced AI tools.

### Enterprise Plan
- Flexible contracts for large organizations
- Features include:
  - **Enterprise-grade security:** Single Sign-On (SSO), audit logs, token management.
  - **Compliance & Governance:** Manage data locations and access with granular controls.
  - **Enhanced Performance:** Advanced compute options including ZeroGPU with increased quotas.
  - **Data Privacy:** Private datasets viewer and private storage per user.
  - **Analytics:** Real-time tracking and dashboard reporting of repository usage.
  - **Billing & Cost Management:** Manage spend through integrated billing for inference providers.

---

## Company Culture

- **Open Innovation:** A foundational belief in open-source technology fueling AI research and development.
- **Community-Centric:** Engages a diverse, global network of contributors and users.
- **Ethical AI:** Commitment to building AI responsibly with transparency and collaboration.
- **Learning & Growth:** A supportive environment where users and employees build skills through active participation in projects and research.

---

## Careers at Hugging Face

Join a passionate team at the frontier of AI technology. Hugging Face seeks individuals eager to make a real-world impact through:

- Research and development of advanced AI/ML models and tools.
- Building scalable platforms that serve a diverse community.
- Collaborating with experts who push the boundaries of machine intelligence.

Explore open positions and become part of a mission-driven company shaping the future of AI.

---

## Connect & Explore

- Website: https://huggingface.co
- GitHub: [Hugging Face GitHub](https://github.com/huggingface)
- Twitter: [@huggingface](https://twitter.com/huggingface)
- LinkedIn: [Hugging Face LinkedIn](https://linkedin.com/company/huggingface)
- Discord Community: Join for discussions and support

---

## Brand Highlights

- Signature Colors: Vibrant Yellow (#FFD21E and #FF9D00), complemented by modern grey (#6B7280)
- Accessible logos and brand assets available for partners and community use.

---

Hugging Face invites everyone— from beginners to seasoned AI professionals—to collaborate, innovate, and build the future of artificial intelligence together. Join the movement and accelerate your machine learning journey today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a vibrant AI community dedicated to building the future of machine learning. As a collaboration platform, it empowers machine learning engineers, scientists, and enthusiasts worldwide to create, share, and discover cutting-edge models, datasets, and applications. Hugging Face serves as a central hub where open-source ML work thrives—fueling innovation across modalities such as text, image, video, audio, and even 3D.

---

## Platform Highlights

- **Massive Model Repository:** Access and contribute to over **2 million models** tailored for various AI tasks.
  
- **Extensive Datasets:** Explore or share from more than **500,000 datasets** curated across diverse domains.
  
- **Interactive Spaces:** Engage with over **1 million applications**, experiment with ML concepts, and collaborate on cutting-edge AI projects in real-time.
  
- **Support for All Modalities:** Work seamlessly with text, images, videos, audio, and 3D data.

- **Open-Source Stack:** Accelerate machine learning workflows with powerful, open-source tools.

- **Collaborative Hub:** Host unlimited public models, datasets, and applications to build your ML portfolio and showcase your work to the global community.

---

## Customers & Community

Hugging Face serves a broad and growing ecosystem of users, including:

- **Machine Learning Engineers & Researchers:** Building state-of-the-art AI models and applications.
- **Data Scientists & Analysts:** Utilizing open datasets for insights and model training.
- **Enterprises:** Leveraging Hugging Face Enterprise offerings to accelerate AI deployment at scale.
- **Academia & Students:** Learning and experimenting with AI in an open and ethical environment.
- **AI Developers & Hobbyists:** Exploring new modalities and contributing to open-source AI innovations.

---

## Company Culture

At Hugging Face, collaboration and openness reign supreme. The community-driven ethos fosters:

- **Open and Ethical AI:** Commitment to transparency, openness, and responsible AI development.
- **Inclusivity and Diversity:** Welcoming contributors from around the globe to share knowledge and create together.
- **Innovation and Learning:** Offering tools and platforms that empower users to accelerate their machine learning journey.
- **Shared Growth:** Encouraging users to build portfolios, gain recognition, and push the boundaries of AI.

---

## Careers & Opportunities

Hugging Face is growing rapidly and offers exciting career paths for individuals passionate about open-source AI and community building, including:

- Machine Learning Engineers
- Research Scientists
- Software Developers
- Community Managers
- Data Specialists
- AI Ethics Advocates

Candidates interested in joining a visionary and community-focused company can look forward to contributing to impactful projects that shape the future of artificial intelligence.

---

## Get Started

Join the future of machine learning today by visiting the Hugging Face platform:

- **Explore models, datasets, applications:** [huggingface.co](https://huggingface.co)
- **Create your profile and share your work**
- **Collaborate with thousands of AI enthusiasts worldwide**

Hugging Face is where machine learning innovation and collaboration meet.

---

## Brand Identity

- **Colors:** Signature yellow (#FFD21E), orange (#FF9D00), gray (#6B7280)
- **Logo:** Friendly and recognizable, symbolizing openness and community spirit
- **Tagline:** *The AI community building the future.*

---

Experience the power of open collaboration. Experience Hugging Face.

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Welcome to Hugging Face: The Happiest Place in AI 🤗

---

## Who We Are

Hugging Face isn’t just a company — it’s *the* AI community building the future of machine learning. Imagine a place where 2 million+ models, hundreds of thousands of datasets, and thousands of machine learning applications hang out, collaborate, and innovate together. Yep, we’re that buzzing hive of AI brilliance.

---

## Why Hugging Face? Because AI Should Be Fun, Collaborative, and Mind-Blowingly Powerful

- **Collaborate like a Pro:** Host and team up on *unlimited* public models, datasets, and applications. Our platform is basically the AI equivalent of a global brain party.
- **Move at the Speed of Thought:** We provide the world’s best open-source ML stack so you can build faster than you can say "neural network."
- **Multi-Modal Magic:** Text? Check. Images? Check. Video? Audio? Heck yes. Even 3D? We got you covered. Dive into any data type and create without limits.
- **Build Your Portfolio & Flex Your ML Skills:** Share your work with the community and make your mark in the AI galaxy.
- **Explore "Spaces":** Try cool apps like a virtual ML intern who’s nothing but a few lines of code — no coffee needed!

---

## Our Star-Studded Lineup Of AI Goodies

- **2 Million+ Models:** From privacy filters that guard your chats to AI that generates 3D art from images, our growing library is your playground.
- **500k+ Datasets:** Curated, cleaned, and ready for your next big breakthrough.
- **1M+ Applications:** Real-world tools and projects created by this vibrant community.

Some fan favorites with social clout:  
- `deepseek-ai/DeepSeek-V4-Pro` (Over 174k users — AI so deep, even your deepest thoughts nod in approval)  
- `openai/privacy-filter` (Because your secrets deserve an AI bodyguard)  
- `Qwen/Qwen3.6-27B` (Massive model? Check. Massive potential? Absolutely.)

---

## Savvy Enterprise & Team Solutions

Got a whole team of AI wizards? Our Team & Enterprise plans start at just $20/user/month with goodies like:

- Single Sign-On (SSO) integration — Because who has time for ten passwords?
- Enterprise-grade security and access controls — Fort Knox, but for your models.
- Audit logs and region control — Stay compliant and control your data kingdom.

We want your AI team to build *securely* and *smoothly* — like a Tesla autopilot but for your ML workflows.

---

## Culture at Hugging Face: Where AI Meets Warm Fuzzies

- **Community First:** We believe open collaboration is the secret sauce. Sharing is caring… and creating!
- **Open Source Evangelists:** Freedom, transparency, and accessibility are in our DNA (next to a love of coffee and cats).
- **Innovate Fearlessly:** We're like the playground for machine learning—try, fail, learn, repeat. No judgement zone.
- **Global & Inclusive:** Whether you're in a startup dorm or a Fortune 500 office, Hugging Face welcomes all humans who dream in algorithms.

---

## Career Opportunities: Join the Machine Learning Party

Love building the future of AI? We’ve got openings for dreamers, coders, researchers, and data magicians alike. Perks:

- Work alongside the cleverest minds in AI
- Work with bleeding-edge open-source tech
- Flexible and remote-friendly vibes
- Contribute to projects that *actually* move the ML needle

---

## Join Us in Hugging the Future 🤗

Whether you’re a data scientist, a research geek, a startup disruptor, or just plain curious, Hugging Face invites you to:

- **Sign Up** for free and explore our universe  
- Collaborate on models, datasets, and apps  
- Accelerate your ML journey with the community that cares

Because the future of AI isn’t just smart — it’s *huggy.*

---

**Website:** [huggingface.co](https://huggingface.co)  
**Come for the AI, stay for the hugs!**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>